# Task 2 — Part A: Local EDA & Preprocessing


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn, joblib, boto3
print("all imports OK — sklearn", sklearn.__version__)

df = pd.read_csv("used_cars.csv")
print("shape:", df.shape)     # expect (4009, 12)
df.head()

all imports OK — sklearn 1.9.0
shape: (4009, 12)


,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price
0,Ford,Utility Police Interceptor Base,2013,"51,000 mi.",E85 Flex Fuel,300.0HP 3.7L V6 Cylinder Engine Flex Fuel Capa...,6-Speed A/T,Black,Black,At least 1 accident or damage reported,Yes,"$10,300"
1,Hyundai,Palisade SEL,2021,"34,742 mi.",Gasoline,3.8L V6 24V GDI DOHC,8-Speed Automatic,Moonlight Cloud,Gray,At least 1 accident or damage reported,Yes,"$38,005"
2,Lexus,RX 350 RX 350,2022,"22,372 mi.",Gasoline,3.5 Liter DOHC,Automatic,Blue,Black,None reported,NaN,"$54,598"
3,INFINITI,Q50 Hybrid Sport,2015,"88,900 mi.",Hybrid,354.0HP 3.5L V6 Cylinder Engine Gas/Electric H...,7-Speed A/T,Black,Black,None reported,Yes,"$15,500"
4,Audi,Q3 45 S line Premium Plus,2021,"9,835 mi.",Gasoline,2.0L I4 16V GDI DOHC Turbo,8-Speed Automatic,Glacier White Metallic,Black,None reported,NaN,"$34,999"


## Inspection of Dataset

In [2]:
# Inspect the raw data BEFORE cleaning
df.info()
print("\n--- Missing values per column ---")
print(df.isnull().sum())
print("\n--- Unique values per column (cardinality) ---")
print(df.nunique())

<class 'pandas.DataFrame'>
RangeIndex: 4009 entries, 0 to 4008
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   brand         4009 non-null   str  
 1   model         4009 non-null   str  
 2   model_year    4009 non-null   int64
 3   milage        4009 non-null   str  
 4   fuel_type     3839 non-null   str  
 5   engine        4009 non-null   str  
 6   transmission  4009 non-null   str  
 7   ext_col       4009 non-null   str  
 8   int_col       4009 non-null   str  
 9   accident      3896 non-null   str  
 10  clean_title   3413 non-null   str  
 11  price         4009 non-null   str  
dtypes: int64(1), str(11)
memory usage: 376.0 KB

--- Missing values per column ---
brand             0
model             0
model_year        0
milage            0
fuel_type       170
engine            0
transmission      0
ext_col           0
int_col           0
accident        113
clean_title     596
price             0
dtype:

## Fix #1  convert milage + price to int

In [3]:
# Fix #1: convert milage and price from text to integers
df["milage"] = df["milage"].str.replace(r"[,]|\s*mi\.", "", regex=True).astype(int)
df["price"]  = df["price"].str.replace(r"[$,]", "", regex=True).astype(int)

print(df[["milage", "price"]].dtypes)
df[["milage", "price"]].describe()

milage    int64
price     int64
dtype: object


,milage,price
count,4009.000000,4.009000e+03
mean,64717.551010,4.455319e+04
std,52296.599459,7.871064e+04
min,100.000000,2.000000e+03
25%,23044.000000,1.720000e+04
50%,52775.000000,3.100000e+04
75%,94100.000000,4.999000e+04
max,405000.000000,2.954083e+06


## Fix #2  fill missing values (fuel_type, accident = mode; clean_title = No→0)

In [4]:
# Fix #2a: fill fuel_type and accident with the mode (most common value)
for col in ["fuel_type", "accident"]:
    df[col] = df[col].fillna(df[col].mode()[0])

print("nulls after 2a:")
print(df[["fuel_type", "accident", "clean_title"]].isnull().sum())

nulls after 2a:
fuel_type        0
accident         0
clean_title    596
dtype: int64


###  Fix #2b: clean_title

In [5]:
# Fix #2b: clean_title — blanks mean "not confirmed clean" -> No -> 0
df["clean_title"] = df["clean_title"].fillna("No").map({"Yes": 1, "No": 0}).astype(int)

print("clean_title values:", df["clean_title"].unique())   # expect [1 0]
print("total nulls left in whole dataset:", df.isnull().sum().sum())  # expect 0

clean_title values: [1 0]
total nulls left in whole dataset: 0


## Fix #3 — Consolidate Messy Categories

Two categorical columns are inconsistent. `fuel_type` contains a `'–'` placeholder (not a real fuel — it means the car has no combustion fuel, i.e. electric). `transmission` has ~62 different spellings of what are really just four transmission types. A model treats every spelling as a separate category, so we must standardize them first.

**Step A:** inspect the raw mess to see exactly what we're fixing.

In [6]:
print("fuel_type values:\n", df["fuel_type"].unique())
print("\ntransmission has", df["transmission"].nunique(), "unique values:\n", df["transmission"].unique())

fuel_type values:
 <StringArray>
[ 'E85 Flex Fuel',       'Gasoline',         'Hybrid',         'Diesel',
 'Plug-In Hybrid',              '–',  'not supported']
Length: 7, dtype: str

transmission has 62 unique values:
 <StringArray>
[                                                     '6-Speed A/T',
                                                '8-Speed Automatic',
                                                        'Automatic',
                                                      '7-Speed A/T',
                                                                'F',
                                                              'A/T',
                                                      '8-Speed A/T',
                                   'Transmission w/Dual Shift Mode',
                                                '9-Speed Automatic',
                                                      '6-Speed M/T',
                                                     '10-Speed A/T',
       

**Step B — clean `fuel_type`:** replace the `'–'` placeholder with `'not supported'` so all "no combustion fuel" cars share one clear label.

**Step C — clean `transmission`:** first drop the few rows where transmission is `'–'` (no usable info). Then normalize the text (lowercase, strip punctuation and extra spaces) and map every variant into one of four buckets: **Automatic, Manual, CVT, Other**.

*Why a function?* The rules are keyword-based (if the text contains "automatic" or "at" → Automatic, etc.). A function keeps that logic readable and easy to defend, instead of writing 62 separate replacements.

In [7]:
import re

# Step B: fuel_type placeholder -> clear label
df["fuel_type"] = df["fuel_type"].replace("–", "not supported")

# Step C: drop useless '–' transmission rows
df = df[df["transmission"] != "–"].copy()

# normalize the text: lowercase, keep only letters/numbers/spaces, collapse spaces
t = (df["transmission"].str.lower()
       .str.replace(r"[^a-z0-9\s]", "", regex=True)
       .str.replace(r"\s+", " ", regex=True)
       .str.strip())

# map each cleaned string into one of 4 buckets
def map_trans(s):
    if "automatic" in s or re.search(r"\bat\b", s): return "Automatic"
    if "manual"    in s or re.search(r"\bmt\b", s): return "Manual"
    if "cvt"       in s:                            return "CVT"
    return "Other"

df["transmission"] = t.apply(map_trans)

print("fuel_type now:", sorted(df["fuel_type"].unique()))
print("\ntransmission now:")
print(df["transmission"].value_counts())

fuel_type now: ['Diesel', 'E85 Flex Fuel', 'Gasoline', 'Hybrid', 'Plug-In Hybrid', 'not supported']

transmission now:
transmission
Automatic    3149
Other         420
Manual        373
CVT            63
Name: count, dtype: int64


## Fix #4 — Feature Engineering
The `engine` column has ~1,146 unique strings — too many to encode. But those strings hide real numbers, so we extract three: horsepower, displacement (litres), cylinders. We also build `car_age` from `model_year`.

In [8]:
# '–' engine means electric car
df["engine"] = df["engine"].replace("–", "Electric")
eng = df["engine"].astype(str)

# pull numbers out of the engine text
df["horsepower"]     = eng.str.extract(r"(\d+\.?\d*)\s*HP")[0].astype(float)
df["displacement_l"] = eng.str.extract(r"(\d+\.?\d*)\s*(?:L\b|Liter)")[0].astype(float)
cyl = eng.str.extract(r"(\d+)\s*Cylinder")[0].astype(float)
cyl = cyl.fillna(eng.str.extract(r"\b[VIWH](\d{1,2})\b")[0].astype(float))  # V6, I4, W12
df["cylinders"] = cyl

# electric / no-fuel cars genuinely have 0 displacement & 0 cylinders
ev = df["fuel_type"].eq("not supported") | eng.str.contains("Electric", case=False)
df.loc[ev, ["displacement_l", "cylinders"]] = df.loc[ev, ["displacement_l", "cylinders"]].fillna(0)

# any remaining gaps -> median (a typical combustion engine)
for col in ["horsepower", "displacement_l", "cylinders"]:
    df[col] = df[col].fillna(df[col].median())
df["cylinders"] = df["cylinders"].round().astype(int)

# car_age is more meaningful than raw year for depreciation
df["car_age"] = (2025 - df["model_year"]).clip(lower=0)

df[["horsepower", "displacement_l", "cylinders", "car_age"]].describe()

,horsepower,displacement_l,cylinders,car_age
count,4005.000000,4005.000000,4005.000000,4005.000000
mean,327.748564,3.496617,5.883645,9.476904
std,109.658137,1.599556,2.139362,6.071196
min,70.000000,0.000000,0.000000,1.000000
25%,268.000000,2.400000,4.000000,5.000000
50%,310.000000,3.500000,6.000000,8.000000
75%,381.000000,4.600000,8.000000,13.000000
max,1020.000000,8.400000,16.000000,33.000000


## Fix #5 — Remove Outliers
Earlier we saw a $2.95M car and a 405k-mile car. These extremes distort scaling and the model. We cut them using the IQR rule on `milage` and `price`.

In [ ]:
# Quartile beacuse we want to remove outliers in milage and price. We will use 2*IQR to remove outliers. and not standard deviation because the data is not normally distributed.
#  We will use 2*IQR to remove outliers because we want to be more conservative in removing outliers.(include luxury cars)

before = len(df)
for col in ["milage", "price"]:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    low, high = q1 - 2 * iqr, q3 + 2 * iqr
    df = df[(df[col] >= low) & (df[col] <= high)]
df = df.reset_index(drop=True)

print(f"removed {before - len(df)} rows -> {len(df)} remain")
df[["milage", "price"]].describe()

removed 217 rows -> 3788 remain


,milage,price
count,3788.000000,3788.000000
mean,66026.504224,34606.690074
std,49241.020851,22734.131776
min,100.000000,2000.000000
25%,25909.000000,16900.000000
50%,55483.500000,29998.000000
75%,95462.500000,46500.000000
max,235000.000000,114900.000000


## Fix #6 — Drop Irrelevant Columns
Remove columns that add noise or are already captured elsewhere, so only useful features go into the model.

In [10]:
df = df.drop(columns=["model", "engine", "ext_col", "int_col", "model_year"])
print(df.columns.tolist())
df.head()

['brand', 'milage', 'fuel_type', 'transmission', 'accident', 'clean_title', 'price', 'horsepower', 'displacement_l', 'cylinders', 'car_age']


,brand,milage,fuel_type,transmission,accident,clean_title,price,horsepower,displacement_l,cylinders,car_age
0,Ford,51000,E85 Flex Fuel,Automatic,At least 1 accident or damage reported,1,10300,300.0,3.7,6,12
1,Hyundai,34742,Gasoline,Automatic,At least 1 accident or damage reported,1,38005,310.0,3.8,6,4
2,Lexus,22372,Gasoline,Automatic,None reported,0,54598,310.0,3.5,6,3
3,INFINITI,88900,Hybrid,Automatic,None reported,1,15500,354.0,3.5,6,10
4,Audi,9835,Gasoline,Automatic,None reported,0,34999,310.0,2.0,4,4


## Step 7 — Train/Test Split (80/20)
Split into features (X) and target (y=price), then into 80% train / 20% test. We split BEFORE scaling/encoding so the test set stays truly unseen.

In [11]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["price"])   # features
y = df["price"]                  # target we want to predict

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("train:", X_train.shape, "| test:", X_test.shape)

train: (3030, 10) | test: (758, 10)


## Step 8 — Preprocessing Pipeline (ColumnTransformer)
Scale the numeric columns and one-hot encode the categorical ones in a single pipeline. Fit on TRAIN only, then apply to test.

In [12]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

num_cols = ["milage", "car_age", "horsepower", "displacement_l", "cylinders"]
cat_cols = ["brand", "fuel_type", "transmission", "accident"]
# clean_title is already 0/1 -> passthrough

pre = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
    ("bin", "passthrough", ["clean_title"]),
], remainder="drop")

X_train_t = pre.fit_transform(X_train)   # FIT on train only
X_test_t  = pre.transform(X_test)        # test: transform only, no fitting

feat_names = pre.get_feature_names_out()
print("features:", X_train_t.shape[1])
print("train:", X_train_t.shape, "| test:", X_test_t.shape)

features: 72
train: (3030, 72) | test: (758, 72)


## Step 9 — Save the Preprocessed CSVs
Save train and test in SageMaker's built-in XGBoost format: label in the FIRST column, and NO header row.

In [13]:
import pandas as pd, joblib

def save_sm(X_arr, y_vals, path):
    out = pd.DataFrame(X_arr, columns=feat_names)
    out.insert(0, "price", y_vals.reset_index(drop=True).values)  # label FIRST
    out.to_csv(path, index=False, header=False)                   # NO header
    return out

train_df = save_sm(X_train_t, y_train, "train.csv")
test_df  = save_sm(X_test_t,  y_test,  "test.csv")

# features-only copy (no label) — used for inference in Part B
pd.DataFrame(X_test_t, columns=feat_names).to_csv("test_features.csv", index=False, header=False)

joblib.dump(pre, "preprocessor.joblib")   # save the fitted pipeline for reuse

print("train.csv:", train_df.shape, "| test.csv:", test_df.shape)
pd.read_csv("train.csv", header=None).head(3).iloc[:, :6]

train.csv: (3030, 73) | test.csv: (758, 73)


,0,1,2,3,4,5
0,39857,-0.342704,-0.588598,-0.122366,0.353427,0.107021
1,29000,-0.770290,-0.754208,-0.892259,-0.899399,-0.855212
2,13000,1.690509,-0.257378,-1.436818,-0.586192,-0.855212


## Part C Prep — Carve a Validation Set from Training Data
Hyperparameter tuning needs a validation set to score each combination against.
We split ONLY train.csv (3,030 rows) into new-train (80%) + validation (20%).
test.csv is untouched, so the final test score stays comparable to Part B.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

full_train = pd.read_csv("train.csv", header=None)   # 3,030 rows, label in col 0
print("original train:", full_train.shape)

# split the ROWS (data is already preprocessed; this is just partitioning)
new_train, validation = train_test_split(
    full_train, test_size=0.20, random_state=42
)

new_train.to_csv("train_c.csv",       index=False, header=False)
validation.to_csv("validation_c.csv", index=False, header=False)
 
print("new train  :", new_train.shape,   "-> train_c.csv")
print("validation :", validation.shape,  "-> validation_c.csv")
print("test.csv    : untouched")

original train: (3030, 73)
new train  : (2424, 73) -> train_c.csv
validation : (606, 73) -> validation_c.csv
test.csv    : untouched
